In [0]:

from pyspark.sql import functions as F

BRONZE = "abfss://lakehouse@saatliqcommerce01dl.dfs.core.windows.net/bronze"

df_marketing = spark.read.parquet(
    f"{BRONZE}/marketing_spend"
)

# 1. Check schema
df_marketing.printSchema()

# 2. Total rows and exact duplicates
print("Total rows:", df_marketing.count())
print(
    "Exact duplicates:",
    df_marketing.count() - df_marketing.distinct().count()
)

# 3. See actual records
display(df_marketing.limit(15))


root
 |-- spend_date: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- campaign: string (nullable = true)
 |-- spend_amount: string (nullable = true)
 |-- clicks: string (nullable = true)

Total rows: 3108
Exact duplicates: 54


spend_date,channel,campaign,spend_amount,clicks
2025-01-01,Email,ADJ_Email_2025_01,-132.98,0
2025-01-01,Email,Email_2025_01,765.63,526
2025-01-01,Google Ads,Google_2025_01,7028.65,1475
2025-01-01,Influencer,Influencer_2025_01,3089.88,656
2025-01-01,Meta Ads,Meta_2025_01,5740.12,1615
2025-01-01,SEO,SEO_2025_01,1303.37,"1,546"
2025-01-02,Email,Email_2025_01,799.09,566
2025-01-02,Google Ads,Google_2025_01,6048.53,1086
2025-01-02,Influencer,Influencer_2025_01,1771.89,365
2025-01-02,Meta Ads,Meta_2025_01,3399.95,906


In [0]:

from pyspark.sql import functions as F

# 1. Remove exact duplicates
raw_count = df_marketing.count()

marketing_dedup = df_marketing.dropDuplicates()

duplicate_count = raw_count - marketing_dedup.count()


# 2. Identify blank spend values
blank_spend = (
    F.col("spend_amount").isNull() |
    (F.trim(F.col("spend_amount")) == "")
)

blank_spend_count = marketing_dedup.filter(blank_spend).count()

marketing_valid_spend = marketing_dedup.filter(~blank_spend)


# 3. Define the 5 standard marketing channels
CHANNEL_MAP = {
    "metaads": "Meta Ads",
    "facebookads": "Meta Ads",
    "googleads": "Google Ads",
    "email": "Email",
    "seo": "SEO",
    "influencer": "Influencer",
    "influencermarketing": "Influencer"
}

channel_map = F.create_map(
    *[F.lit(x) for pair in CHANNEL_MAP.items() for x in pair]
)

# Ignore casing, spaces and punctuation in lookup keys
channel_key = F.regexp_replace(
    F.lower(F.trim(F.col("channel"))),
    r"[^a-z]",
    ""
)


# 4. Clean dates, channels, spend and clicks
marketing_clean = (
    marketing_valid_spend
    .withColumn("channel", channel_map[channel_key])
    .withColumn("campaign", F.trim(F.col("campaign")))
    .withColumn(
        "spend_date",
        F.expr("try_cast(trim(spend_date) AS DATE)")
    )
    .withColumn(
        "spend_amount",
        F.expr("""
            try_cast(
                regexp_replace(trim(spend_amount), ',', '')
                AS DECIMAL(18,2)
            )
        """)
    )
    .withColumn(
        "clicks",
        F.expr("""
            try_cast(
                regexp_replace(trim(clicks), ',', '')
                AS BIGINT
            )
        """)
    )
)

print("Raw rows:", raw_count)
print("Exact duplicates removed:", duplicate_count)
print("Blank spend rows removed:", blank_spend_count)
print("Cleaned rows:", marketing_clean.count())

display(marketing_clean.limit(10))


Raw rows: 3108
Exact duplicates removed: 54
Blank spend rows removed: 17
Cleaned rows: 3037


spend_date,channel,campaign,spend_amount,clicks
2025-01-01,Google Ads,Google_2025_01,7028.65,1475
2025-01-06,SEO,SEO_2025_01,909.83,1161
2025-01-16,Google Ads,Google_2025_01,6025.04,1246
2025-01-22,Google Ads,Google_2025_01,5236.89,990
2025-01-23,Influencer,Influencer_2025_01,1615.83,320
2025-01-23,Meta Ads,Meta_2025_01,5390.47,1338
2025-01-27,Meta Ads,Meta_2025_01,2583.09,835
2025-02-02,Email,Email_2025_02,709.91,528
2025-02-12,Google Ads,Google_2025_02,6951.04,1496
2025-02-20,Influencer,Influencer_2025_02,3616.10,576


In [0]:

# Validate the cleaned data
invalid_rows = marketing_clean.filter(
    F.col("spend_date").isNull() |
    F.col("channel").isNull() |
    F.col("spend_amount").isNull() |
    F.col("clicks").isNull() |
    (F.col("clicks") < 0)
).count()

print("Invalid rows:", invalid_rows)

# If unknown channels exist, show their raw values
if invalid_rows > 0:
    display(
        marketing_valid_spend
        .select("channel")
        .distinct()
        .withColumn("mapped", channel_map[channel_key])
        .filter(F.col("mapped").isNull())
    )
    raise ValueError("Marketing validation failed")

print("Marketing validation PASSED!")

# Full snapshot overwrite
TARGET = "atliq_dl.silver.marketing_spend"

marketing_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(TARGET)

# Verify persisted Silver table
display(
    spark.table(TARGET).agg(
        F.count("*").alias("silver_rows"),
        F.countDistinct("channel").alias("channels"),
        F.sum("spend_amount").alias("total_spend"),
        F.sum("clicks").alias("total_clicks")
    )
)


Invalid rows: 0
Marketing validation PASSED!


silver_rows,channels,total_spend,total_clicks
3037,5,8920850.16,3274996


In [0]:
%run ./utils_dq

In [0]:
raw = spark.read.parquet(f"{BRONZE}/marketing_spend")
silver = spark.table("atliq_dl.silver.marketing_spend")

unique_rows = raw.dropDuplicates()

# Accepted channel names from our clean Silver table
valid_channels = [
    row["channel"]
    for row in silver.select("channel").distinct().collect()
]

log_dq("marketing_spend", "exact_duplicates_removed",
       raw.count() - unique_rows.count())

log_dq("marketing_spend", "blank_spend_removed",
       unique_rows.filter(
           F.col("spend_amount").isNull() |
           (F.trim(F.col("spend_amount").cast("string")) == "")
       ).count())

log_dq("marketing_spend", "channel_standardized",
       unique_rows.filter(
           F.col("channel").isNotNull() &
           ~F.col("channel").isin(*valid_channels)
       ).count())

log_dq("marketing_spend", "clicks_format_cleaned",
       unique_rows.filter(
           F.col("clicks").cast("string").contains(",")
       ).count())

log_dq("marketing_spend", "negative_spend_corrections_retained",
       silver.filter(F.col("spend_amount") < 0).count())

display(
    spark.table("atliq_dl.silver.dq_log")
    .filter("table_name = 'marketing_spend'")
    .orderBy("rule")
)

run_date,table_name,rule,rows_affected
2026-11-01,marketing_spend,blank_spend_removed,17
2026-10-31,marketing_spend,blank_spend_removed,17
2026-10-31,marketing_spend,channel_standardized,473
2026-11-01,marketing_spend,channel_standardized,473
2026-10-31,marketing_spend,clicks_format_cleaned,451
2026-11-01,marketing_spend,clicks_format_cleaned,451
2026-11-01,marketing_spend,exact_duplicates_removed,54
2026-10-31,marketing_spend,exact_duplicates_removed,54
2026-10-31,marketing_spend,negative_spend_corrections_retained,44
2026-11-01,marketing_spend,negative_spend_corrections_retained,44
